# Taxi Trip Data Processing
Monthly chunk pipeline for NYC TLC yellow taxi data.

In [26]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa

In [27]:
# --- Configuration ---
YEAR = 2022
TAXI_TYPE = 'yellow'
INPUT_DIR = Path('../raw')
OUTPUT_DIR = Path('../processed')
REPORTS_DIR = Path('../reports')

# --- Add constants for data cleaning ---
FILTER_SPEED_ABOVE = 100
TRIP_LENGTH_FILTER_MIN = 1  # in minutes
TRIP_LENGTH_FILTER_MAX = 240  # in minutes

In [28]:
def clean_data(df: pd.DataFrame, initial_rows: int) -> (pd.DataFrame, pd.DataFrame):
    """
    Applies a series of data cleaning and quality assurance steps to a chunk of taxi trip data.

    Args:
        df: The raw taxi trip DataFrame chunk.
        initial_rows: The number of rows in the original, unfiltered DataFrame for percentage calculation.

    Returns:
        A tuple containing the cleaned DataFrame and a summary of the QA checks for the chunk.
    """
    qa_summary = {}

    # Convert to datetime and calculate trip duration
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

    # Define and apply filters
    cost_cols = ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']
    filters = {
        'trip_outside_year': (df['tpep_pickup_datetime'].dt.year != YEAR) | (df['tpep_dropoff_datetime'].dt.year != YEAR),
        'invalid_trip_duration': df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime'],
        'zero_or_negative_trip_distance': df['trip_distance'] <= 0,
        'zero_passenger_count': df['passenger_count'] == 0,
        'negative_cost_values': (df[cost_cols] < 0).any(axis=1),
        'outlier_trip_duration': (df['trip_duration_minutes'] < TRIP_LENGTH_FILTER_MIN) | (df['trip_duration_minutes'] > TRIP_LENGTH_FILTER_MAX)
    }

    for name, mask in filters.items():
        qa_summary[name] = mask.sum()
        df = df[~mask]

    # Calculate trip speed and apply speed filter
    non_zero_duration_mask = df['trip_duration_minutes'] > 0
    df['trip_speed_mph'] = 0.0
    df.loc[non_zero_duration_mask, 'trip_speed_mph'] = df.loc[non_zero_duration_mask, 'trip_distance'] / (df.loc[non_zero_duration_mask, 'trip_duration_minutes'] / 60)
    
    speed_filter = df['trip_speed_mph'] > FILTER_SPEED_ABOVE
    qa_summary['outlier_trip_speed'] = speed_filter.sum()
    df = df[~speed_filter]

    summary_df = pd.DataFrame.from_dict(qa_summary, orient='index', columns=['records_removed'])
    return df, summary_df

In [29]:
def feature_engineering_and_kpis(df: pd.DataFrame, year: int):
    """
    Engineers time-based features and calculates daily, weekly, and monthly KPIs.
    This function operates on a concatenated DataFrame containing only the columns
    necessary for KPI calculation, making it memory-efficient.

    Args:
        df: The taxi trip DataFrame with columns for KPI calculation.
        year: The year of the data, used for naming output files.
    """
    print("Starting feature engineering and KPI calculation on combined data...")

    # 1. Engineer time-based features
    df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
    df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
    df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek
    df['day_name'] = df['tpep_pickup_datetime'].dt.day_name()
    df['week_of_year'] = df['tpep_pickup_datetime'].dt.isocalendar().week
    df['month'] = df['tpep_pickup_datetime'].dt.month
    df['month_name'] = df['tpep_pickup_datetime'].dt.month_name()

    # 2. Define KPI calculation logic
    def calculate_kpis(grouped_df):
        return grouped_df.agg(
            total_trips=('tpep_pickup_datetime', 'count'),
            p50_trip_duration=('trip_duration_minutes', lambda x: x.quantile(0.5)),
            p95_trip_duration=('trip_duration_minutes', lambda x: x.quantile(0.95)),
            p50_trip_speed=('trip_speed_mph', lambda x: x.quantile(0.5))
        )

    # 3. Calculate KPIs by different time aggregations
    # Daily KPIs
    daily_kpis = calculate_kpis(df.groupby('pickup_date'))
    daily_kpis.to_csv(OUTPUT_DIR / f'kpi_daily_{year}.csv')
    print(f"Daily KPIs saved to {OUTPUT_DIR / f'kpi_daily_{year}.csv'}")

    # Weekly KPIs
    weekly_kpis = calculate_kpis(df.groupby('week_of_year'))
    weekly_kpis.to_csv(OUTPUT_DIR / f'kpi_weekly_{year}.csv')
    print(f"Weekly KPIs saved to {OUTPUT_DIR / f'kpi_weekly_{year}.csv'}")

    # Monthly KPIs
    monthly_kpis = calculate_kpis(df.groupby('month'))
    monthly_kpis.to_csv(OUTPUT_DIR / f'kpi_monthly_{year}.csv')
    print(f"Monthly KPIs saved to {OUTPUT_DIR / f'kpi_monthly_{year}.csv'}")

In [30]:
def combine_parquet_files(files_to_combine, output_path):
    """Combines multiple Parquet files into a single file memory-efficiently."""
    writer = None
    try:
        for i, path in enumerate(files_to_combine):
            table = pq.read_table(str(path))
            if i == 0:
                writer = pq.ParquetWriter(str(output_path), table.schema)
            writer.write_table(table)
    finally:
        if writer:
            writer.close()

In [31]:
def main():
    """Main execution function to run the processing pipeline in chunks."""
    OUTPUT_DIR.mkdir(exist_ok=True)
    REPORTS_DIR.mkdir(exist_ok=True)
    temp_dir = OUTPUT_DIR / 'temp_cleaned'
    temp_dir.mkdir(exist_ok=True)

    try:
        trip_files = sorted(list(INPUT_DIR.glob(f'{TAXI_TYPE}_tripdata_{YEAR}-*.parquet')))
        if not trip_files:
            print(f"Error: No Parquet files found in '{INPUT_DIR}' for {YEAR}.\nPlease run the download script first.")
            return

        def process_chunk(file_path):
            print(f"Processing chunk: {file_path.name}...")
            df_chunk = pd.read_parquet(file_path)
            initial_rows = len(df_chunk)
            cleaned_df, qa_summary = clean_data(df_chunk, initial_rows)
            
            temp_output_path = temp_dir / f'cleaned_{file_path.name}'
            cleaned_df.to_parquet(temp_output_path, index=False)
            
            kpi_cols = ['tpep_pickup_datetime', 'trip_duration_minutes', 'trip_speed_mph']
            return initial_rows, qa_summary, cleaned_df[kpi_cols], temp_output_path

        results = [process_chunk(fp) for fp in trip_files]
        total_initial_rows = sum(r[0] for r in results)
        all_qa_summaries = [r[1] for r in results]
        kpi_cols_list = [r[2] for r in results]
        cleaned_chunk_paths = [r[3] for r in results]

        # --- Aggregate QA reports ---
        print("Aggregating QA reports...")
        final_qa_summary = pd.concat(all_qa_summaries).groupby(level=0).sum()
        final_qa_summary['percentage_of_total'] = (final_qa_summary['records_removed'] / total_initial_rows) * 100
        qa_report_path = REPORTS_DIR / f'qa_summary_{YEAR}.csv'
        final_qa_summary.to_csv(qa_report_path)
        print(f"QA summary saved to {qa_report_path}")

        # --- Combine chunks for KPI calculation ---
        print("Combining data for KPI calculations...")
        kpi_source_df = pd.concat(kpi_cols_list, ignore_index=True)

        # --- Feature Engineering and KPI Calculation ---
        feature_engineering_and_kpis(kpi_source_df, YEAR)

        # --- Combine cleaned chunks into a single file ---
        print("Combining cleaned data into final Parquet file...")
        final_cleaned_path = OUTPUT_DIR / f'cleaned_{TAXI_TYPE}_tripdata_{YEAR}.parquet'
        combine_parquet_files(cleaned_chunk_paths, final_cleaned_path)
        print(f"Final cleaned data saved to {final_cleaned_path}")

    finally:
        # --- Cleanup temporary files ---
        if temp_dir.exists():
            print("Cleaning up temporary files...")
            for temp_file in temp_dir.iterdir():
                temp_file.unlink()
            temp_dir.rmdir()

In [32]:
if __name__ == '__main__':
    main()

Processing chunk: yellow_tripdata_2022-01.parquet...


C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]


Processing chunk: yellow_tripdata_2022-02.parquet...


C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]


Processing chunk: yellow_tripdata_2022-03.parquet...


...

Processing chunk: yellow_tripdata_2022-11.parquet...


C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]


Processing chunk: yellow_tripdata_2022-12.parquet...


C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]
C:\Users\My PC\AppData\Local\Temp\ipykernel_17120\932287707.py:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~mask]


Aggregating QA reports...
QA summary saved to ..\reports\qa_summary_2022.csv
Combining data for KPI calculations...
Starting feature engineering and KPI calculation on combined data...
Daily KPIs saved to ..\processed\kpi_daily_2022.csv
Weekly KPIs saved to ..\processed\kpi_weekly_2022.csv
Monthly KPIs saved to ..\processed\kpi_monthly_2022.csv
Combining cleaned data into final Parquet file...
Final cleaned data saved to ..\processed\cleaned_yellow_tripdata_2022.parquet
Cleaning up temporary files...


In [ ]:
df = pd.read_parquet("..\processed\cleaned_yellow_tripdata_2022.parquet").sample(10000)

In [19]:
df = sorted(df['tpep_pickup_datetime'])

In [21]:
df

[Timestamp('2001-01-01 00:03:14'),
 Timestamp('2001-01-01 00:27:45'),
 Timestamp('2001-01-01 01:23:51'),
 Timestamp('2001-08-23 05:34:45'),
 Timestamp('2002-10-21 00:13:16'),
 Timestamp('2002-10-21 00:27:54'),
 Timestamp('2002-10-21 05:50:56'),
 Timestamp('2002-10-21 05:54:54'),
 Timestamp('2002-10-21 08:06:54'),
 Timestamp('2002-10-21 08:31:53'),
 Timestamp('2002-10-21 09:07:51'),
 Timestamp('2002-10-21 09:50:44'),
 Timestamp('2002-10-21 09:52:38'),
 Timestamp('2002-10-21 09:55:45'),
 Timestamp('2002-10-21 10:18:52'),
 Timestamp('2002-10-21 10:36:45'),
 Timestamp('2002-10-21 11:21:37'),
 Timestamp('2002-10-21 12:05:23'),
 Timestamp('2002-10-21 12:06:16'),
 Timestamp('2002-10-21 13:08:49'),
 Timestamp('2002-10-21 15:46:22'),
 Timestamp('2002-10-21 15:55:48'),
 Timestamp('2002-10-21 15:58:33'),
 Timestamp('2002-10-21 16:19:19'),
 Timestamp('2002-10-21 16:32:13'),
 Timestamp('2002-10-22 07:17:32'),
 Timestamp('2002-10-22 08:55:12'),
 Timestamp('2002-10-22 09:11:17'),
 Timestamp('2002-10-